### **Hidden Markov Model Demo With 1 Device**

##### **What is a Hidden Markov Model?**

Hidden Markov Model (HMM) is a statistical model used to predict sequences of data by inferring "hidden" internal states from observable events.

In NILM: 

| Component | Value | Note |
|:-|:-|:-|
| **Hidden Status** | `Appliance1 {OFF, ON}` | *← don't see it directly* |
| **Observe** | `Aggregate = 750W` | *← only see this total* |

HMM has three main components:

| Component | Intuitive Meaning | Example / Parameters |
|:---|:---|:---|
| **1. Initial Probability** ($\pi$) | "At the very beginning, what is the probability (%) that the appliance is ON?" | $\pi = [0.7, 0.3]$<br>$\rightarrow$ 70% OFF, 30% ON initially. |
| **2. Transition Matrix** ($A$) | "If it is currently ON, what is the probability (%) it stays ON in the next step?" | $A = \begin{bmatrix} 0.98 & 0.02 \\ 0.05 & 0.95 \end{bmatrix}$<br><br>$\rightarrow$ **If OFF:** 98% stays OFF, 2% turns ON.<br>$\rightarrow$ **If ON:** 5% turns OFF, 95% stays ON. |
| **3. Emission Distribution** ($B$) | "When the appliance is ON, how much power (W) does it consume?" | E.g., Gaussian distribution:<br>$\rightarrow$ $\mu_{OFF} = 0W, \sigma = 10$<br>$\rightarrow$ $\mu_{ON} = 2000W, \sigma = 200$ |


##### **Factorial Hidden Markov Model and Hidden Markov Model**

Standard HMM:

+ 1 hidden sequence $\to$ monitors 1 thing

FHMM (Factorial HMM):

+ K parallel hidden sequences $\to$ monitors K things simultaneously

Problem:

+ K = 9 devices, each running its own HMM

+ Appliance1_state: OFF $\to$ OFF $\to$ ON $\to$ ON $\to$ OFF...

+ Appliance2_state: ON $\to$ ON $\to$ ON $\to$ OFF $\to$ OFF...

+ ...

+ Appliance9_state: OFF $\to$ OFF $\to$ OFF $\to$ ON $\to$ ON...

Observation: Aggregate = $\sum$(power of devices that are currently ON)

**Import**

In [1]:
import sys
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

**Set the project path**

In [2]:
base_dir = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(base_dir)
print(f"base_dir = {base_dir}")

data_dir = os.path.join(base_dir, 'data', 'processed_data')
print(f"data_dir = {data_dir}")

print(f"The files in data_dir: {os.listdir(data_dir)}")

base_dir = d:\Document\2025.2\IT3191E_Machine_Learning\ML_FinalProject_NILM_REFIT
data_dir = d:\Document\2025.2\IT3191E_Machine_Learning\ML_FinalProject_NILM_REFIT\data\processed_data
The files in data_dir: ['House2_full.csv', 'House2_part1.csv', 'House2_part2.csv', 'House2_part3.csv', 'House2_part4.csv', 'House2_part5.csv']


**Combine 5 CSV files into one `DataFrame`**

In [4]:
print("Reading data...")
dfs = []
for i in range(1, 6):  # i = 1, 2, 3, 4, 5
    filepath = os.path.join(data_dir, f'House2_part{i}.csv')
    print(f"  Reading House2_part{i}.csv...")
    df_part = pd.read_csv(filepath)
    dfs.append(df_part)

df = pd.concat(dfs, axis=0, ignore_index=True)
print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")
print(f"Column names: {df.columns.tolist()}")

Reading data...
  Reading House2_part1.csv...
  Reading House2_part2.csv...
  Reading House2_part3.csv...
  Reading House2_part4.csv...
  Reading House2_part5.csv...
Number of rows: 5,733,526
Number of columns: 12
Column names: ['Time', 'Unix', 'Aggregate', 'Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']


**Save the merged df to a `.csv` file**

In [5]:
full_path = os.path.join(data_dir, 'House2_full.csv')

if not os.path.exists(full_path):
    print("Saving House2_full.csv...")
    df.to_csv(full_path, index=False)
    print(f"Saved at: {full_path}")
else:
    print(f"The file already exists")
    print("Reloading from House2_full.csv...")
    df = pd.read_csv(full_path)
    print(f"Loaded: {len(df):,} rows")


The file already exists
Reloading from House2_full.csv...
Loaded: 5,733,526 rows


**Process the `Time` column**

In [6]:
df['Time'] = pd.to_datetime(df['Time'])

print(f"Time data type: {df['Time'].dtype}") 
print(f"Start time: {df['Time'].min()}")
print(f"End time: {df['Time'].max()}")
print(f"Time interval: {df['Time'].max() - df['Time'].min()}")

time_diff = df['Time'].diff().dropna()
print(f"Mean distance between samples: {time_diff.mean()}")
print(f"Most common distance: {time_diff.mode()[0]}")

Time data type: datetime64[us]
Start time: 2013-09-17 22:08:11
End time: 2015-05-28 08:05:43
Time interval: 617 days 09:57:32
Mean distance between samples: 0 days 00:00:09.303988
Most common distance: 0 days 00:00:07


**Calculate `Appliance_Others`**

In [7]:
appliance_cols = ['Appliance1', 'Appliance2', 'Appliance3', 
                  'Appliance4', 'Appliance5', 'Appliance6',
                  'Appliance7', 'Appliance8', 'Appliance9']

df['sum_9_appliances'] = df[appliance_cols].sum(axis=1)

df['Appliance_Others'] = df['Aggregate'] - df['sum_9_appliances']

negative_count = (df['Appliance_Others'] < 0).sum()
print(f"Number of rows with Appliance_Others < 0 (to be deleted): {negative_count:,}")

df = df[df['Appliance_Others'] >= 0].copy()
print(f"The {negative_count:,} line has been removed. The current minimum value is: {df['Appliance_Others'].min()}")

df.drop(columns=['sum_9_appliances'], inplace=True)

print(f"Appliance_Others summarizes:")
print(df['Appliance_Others'].describe())

Number of rows with Appliance_Others < 0 (to be deleted): 28,444
The 28,444 line has been removed. The current minimum value is: 0
Appliance_Others summarizes:
count    5.705082e+06
mean     3.214029e+02
std      9.227989e+02
min      0.000000e+00
25%      8.300000e+01
50%      1.120000e+02
75%      2.300000e+02
max      2.438700e+04
Name: Appliance_Others, dtype: float64


**HMM demo with 1 device**

In [9]:
from hmmlearn import hmm

col = 'Appliance1'

observations = df[col].values.reshape(-1, 1)
print(f"Shape of observations: {observations.shape}")
print(f"Example of the first 5 values: {observations[:5, 0]}")

Shape of observations: (5705082, 1)
Example of the first 5 values: [88 88 88 88 88]


**Create and train HMM**

In [10]:
model_app1 = hmm.GaussianHMM(
    n_components=2,          # number of states: 2 = {OFF, ON}
    covariance_type="diag",  # covariance matrix
    n_iter=100,              # maximum number of iterations of the EM algorithm
    random_state=42
)

obs_train = observations[:100_000]
print("Currently training HMM for Appliance1...")
model_app1.fit(obs_train)
print("Train complete!")

Currently training HMM for Appliance1...
Train complete!


**View the results of the model learned**

In [13]:
print("1. Mean values ​​of each state:")
for i, mean in enumerate(model_app1.means_):
    print(f"   Status {i}: {mean[0]:.2f} W")

print("\n2. Transmat:")
print("   (row = current state, column = next state)")
print(f"   {model_app1.transmat_}")

print("\n3. Startprob:")
print(f"   {model_app1.startprob_}")


1. Mean values ​​of each state:
   Status 0: 4.24 W
   Status 1: 85.52 W

2. Transmat:
   (row = current state, column = next state)
   [[0.99794849 0.00205151]
 [0.00339554 0.99660446]]

3. Startprob:
   [1.90564239e-72 1.00000000e+00]


**Predicting hidden states**

In [15]:
obs_test = observations[100_000:110_000]  # lấy 10,000 điểm để test
states = model_app1.predict(obs_test)

print(f"The first 10 states: {states[:10]}")
print(f"% time in state 0: {(states==0).mean()*100:.1f}%")
print(f"% time in state 1: {(states==1).mean()*100:.1f}%")

predicted_power = np.array([model_app1.means_[s, 0] for s in states])
actual_power = obs_test[:, 0]

print(f"\nActual (average) power output: {actual_power.mean():.2f}W")
print(f"Predicted (average) power output: {predicted_power.mean():.2f}W")


The first 10 states: [0 0 0 0 0 0 0 0 0 0]
% time in state 0: 60.4%
% time in state 1: 39.6%

Actual (average) power output: 37.88W
Predicted (average) power output: 36.40W
